In [1]:
import json
import random
import string
import nltk
import numpy as np
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Download necessary NLTK data
nltk.download("punkt")
nltk.download("wordnet")

# Initialize the lemmatizer
lemmatizer = WordNetLemmatizer()

# Sample data with medical intents and responses
data = {
    "intents": [
        {"tag": "digestive_issues",
         "patterns": ["I have digestive issues", "What can help with nausea?", "I'm feeling bloated"],
         "responses": ["Ginger Tea: Brew fresh ginger slices in hot water to alleviate nausea and indigestion. Ginger is known for its calming properties and can help soothe the stomach, reducing discomfort.",
                       "Apple Cider Vinegar: Mixing 1-2 tablespoons of apple cider vinegar in a glass of water can aid digestion and relieve heartburn. It may also help balance the stomach's acidity levels."]
         },
        {"tag": "cold_and_flu",
         "patterns": ["I have a cold", "What can I take for flu symptoms?", "How to soothe a sore throat?"],
         "responses": ["Honey and Lemon: Mixing honey and lemon juice in warm water is a soothing remedy for sore throats and coughs. Honey has antimicrobial properties and can help coat the throat, while lemon adds vitamin C.",
                       "Steam Inhalation: Inhaling steam from a bowl of hot water with a few drops of eucalyptus oil can help relieve nasal congestion and clear your sinuses, making it easier to breathe."]
         },
        {"tag": "skin_care",
         "patterns": ["How to treat sunburn?", "Best remedies for acne", "What can I use for irritated skin?"],
         "responses": ["Aloe Vera Gel: Applying fresh aloe vera gel to sunburned or irritated skin provides soothing relief. Aloe vera has anti-inflammatory properties that help reduce redness and promote healing.",
                       "Honey and Cinnamon Mask: Mixing honey with cinnamon creates a paste that can help treat acne due to its antibacterial properties, which may help prevent breakouts."]
         },
        {"tag": "headaches_and_migraines",
         "patterns": ["I have a headache", "What can help with migraines?", "Tips for relieving headache pain?"],
         "responses": ["Peppermint Oil: Applying diluted peppermint oil to your temples provides a cooling sensation that may help relieve headache pain. Its menthol content can improve blood flow and reduce tension.",
                       "Cold Compress: Using a cold pack on your forehead or neck can help alleviate headache pain by numbing the area and reducing inflammation."]
         },
        {"tag": "stress_and_anxiety",
         "patterns": ["How to reduce stress?", "What can help with anxiety?", "Natural remedies for relaxation"],
         "responses": ["Chamomile Tea: Drinking chamomile tea is an excellent way to promote relaxation and reduce anxiety. Chamomile is known for its calming effects, which can help ease stress.",
                       "Lavender Oil: Using lavender essential oil in a diffuser or as a topical application may provide calming effects and reduce anxiety levels."]
         },
        {"tag": "greetings",
         "patterns": ["Hi", "Hello", "Good morning", "Good evening", "How are you?"],
         "responses": ["Hello! How can I assist you today? I'm here to help you with your health questions.",
                       "Hi there! Feel free to ask me anything related to health and wellness. I'm here for you!",
                       "Good day! If you have any health concerns or questions, I'm just a message away."]
         },
        # Add more intents here...
        {"tag": "goodbye",
         "patterns": ["bye", "g2g", "see ya", "adios", "cya"],
         "responses": ["Take care! Stay healthy!", "Goodbye! Stay safe!", "See you later!"]
         }
    ]
}

# Initialize words, classes, doc_x, and doc_y
words = []
classes = []
doc_x = []
doc_y = []

# Tokenize and process the data
for intent in data["intents"]:
    for pattern in intent["patterns"]:
        tokens = nltk.word_tokenize(pattern)
        words.extend(tokens)
        doc_x.append(pattern)
        doc_y.append(intent["tag"])
    if intent["tag"] not in classes:
        classes.append(intent["tag"])

# Process the words list, removing punctuation
words = [lemmatizer.lemmatize(word.lower()) for word in words if word not in string.punctuation]
words = sorted(set(words))
classes = sorted(set(classes))

# Preparing the training data
training = []
out_empty = [0] * len(classes)
for idx, doc in enumerate(doc_x):
    bow = []
    text = lemmatizer.lemmatize(doc.lower())
    for word in words:
        bow.append(1) if word in text else bow.append(0)
    output_row = list(out_empty)
    output_row[classes.index(doc_y[idx])] = 1
    training.append([bow, output_row])

random.shuffle(training)
training = np.array(training, dtype=object)

# Split the training data into input (X) and output (y)
train_X = np.array(list(training[:, 0]))
train_y = np.array(list(training[:, 1]))

# Create a Sequential model
model = Sequential()
model.add(Dense(128, input_shape=(len(train_X[0]),), activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(len(train_y[0]), activation='softmax'))

# Create the Adam optimizer with a specified learning rate
adam = tf.keras.optimizers.Adam(learning_rate=0.01)

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

print(model.summary())

# Train the model
model.fit(x=train_X, y=train_y, epochs=500, verbose=1)

# Function to clean user input text
def clean_text(text):
    tokens = nltk.word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return tokens

# Function to convert user input into a bag-of-words representation
def bag_of_words(text, vocab):
    tokens = clean_text(text)
    bow = [0] * len(vocab)
    for w in tokens:
        for idx, word in enumerate(vocab):
            if word == w:
                bow[idx] = 1
    return np.array(bow)

# Function to predict the user's intent based on their input
def pred_class(text, vocab, labels):
    bow = bag_of_words(text, vocab)
    result = model.predict(np.array([bow]))[0]
    thresh = 0.2
    y_pred = [[idx, res] for idx, res in enumerate(result) if res > thresh]
    y_pred.sort(key=lambda x: x[1], reverse=True)
    return_list = []
    for r in y_pred:
        return_list.append(labels[r[0]])
    return return_list

# Function to get a response from the predicted intent
def get_response(intents_list, intents_json):
    tag = intents_list[0]
    list_of_intents = intents_json["intents"]
    for i in list_of_intents:
        if i["tag"] == tag:
            result = random.choice(i["responses"])
            break
    return result


[nltk_data] Downloading package punkt to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\maria
[nltk_data]     selciya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
C:\Users\maria selciya\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         7,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,751 (61.53 KB)

 Trainable params: 15,751 (61.53 KB)

 Non-trainable params: 0 (0.00 B)

None
Epoch 1/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1200 - loss: 1.9353
Epoch 2/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - accuracy: 0.3600 - loss: 1.8875
Epoch 3/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.3600 - loss: 1.8470
Epoch 4/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.3600 - loss: 1.7404
Epoch 5/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.6400 - loss: 1.4510
Epoch 6/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - accuracy: 0.8400 - loss: 1.3240
Epoch 7/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.8000 - loss: 1.1968
Epoch 8/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.6800 - loss: 1.2602
Epoch 9/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.8800 - loss: 0.9884
Epoch 10/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.8000 - loss: 0.9601
Epoch 11/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.8800 - loss: 0.6593
Epoch 12/500
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.9200 

In [ ]:
while True:
    message = input("")
    
    # Break condition: if user types exit, quit, or bye
    if message.lower() in ["exit", "quit", "bye","catch you later","i'm leaving"," i need to go"]:
        print("Goodbye!! Have a nice day....Stay Super positive!!!")
        break

    # Use pred_class to get intents and respond
    intents = pred_class(message, words, classes)
    result = get_response(intents, data)
    print(result)

 hii


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
Take care! Stay healthy!


 fever


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
See you later!


 cough


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
See you later!


 dry skin


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Aloe Vera Gel: Applying fresh aloe vera gel to sunburned or irritated skin provides soothing relief. Aloe vera has anti-inflammatory properties that help reduce redness and promote healing.


 aloe vera


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
See you later!
